In [3]:
 %pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os

In [9]:
import getpass

In [10]:
groq_key = os.getenv("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key (hidden): ")

Paste your Groq API key (hidden):  ········


In [11]:
from openai import OpenAI

In [13]:
groq_client = OpenAI(api_key=groq_key, base_url="https://api.groq.com/openai/v1")

In [14]:
for model in groq_client.models.list().data:
    print (model.id)

canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
whisper-large-v3
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b


In [15]:
GROQ_MODEL = "openai/gpt-oss-120b"

In [16]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "In two sentences, what is a Kubernetes liveness probe?"}],
)

In [17]:
response

ChatCompletion(id='chatcmpl-1365ae0a-1104-4b27-b308-3df8184e0b32', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A Kubernetes liveness probe is a periodic health check that the kubelet runs on a container to determine whether the application inside is still running correctly. If the probe fails repeatedly, Kubernetes automatically restarts the container to recover from the unhealthy state.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks: "In two sentences, what is a Kubernetes liveness probe?" Need to answer in two sentences, concise. Provide definition. Ensure two sentences.'))], created=1794276007, model='openai/gpt-oss-120b', object='chat.completion', metadata=None, moderation=None, service_tier='on_demand', system_fingerprint='fp_05e76be18c', usage=CompletionUsage(completion_tokens=91, prompt_tokens=83, total_tokens=174, completion_tokens_details=Comp

In [18]:
print(response.choices[0].message.content)
print()
print("input tokens :", response.usage.prompt_tokens)
print("output tokens:", response.usage.completion_tokens)

A Kubernetes liveness probe is a periodic health check that the kubelet runs on a container to determine whether the application inside is still running correctly. If the probe fails repeatedly, Kubernetes automatically restarts the container to recover from the unhealthy state.

input tokens : 83
output tokens: 91


In [19]:
local_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1") 

In [20]:
for model in local_client.models.list().data:
    print(model.id)

llama3.2:latest


In [21]:
LOCAL_MODEL = "llama3.2:latest"  

In [22]:
response = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[{"role": "user", "content": "In two sentences, what is a Kubernetes liveness probe?"}],
)

print(response.choices[0].message.content)

A Kubernetes liveness probe is a test that runs periodically to determine if a container is running correctly and responding to requests. If the probe fails, it can trigger a restart of the container, ensuring that the application is healthy and functioning as expected.


In [23]:
tickets = [
    "Checkout page returns HTTP 500 after the release",
    "Node ran out of disk space and pods are being evicted",
    "Replication lag on the orders replica keeps growing",
    "Backup of the orders database failed: cannot connect to the database",
]

In [24]:
template = """Which team should handle this incident ticket?
Choose one: application, infrastructure, database, backup.
Answer with the team name only.

Ticket: {ticket}"""

In [25]:
for ticket in tickets:
    messages = [{"role": "user", "content": template.format(ticket=ticket)}]
    print (messages)

[{'role': 'user', 'content': 'Which team should handle this incident ticket?\nChoose one: application, infrastructure, database, backup.\nAnswer with the team name only.\n\nTicket: Checkout page returns HTTP 500 after the release'}]
[{'role': 'user', 'content': 'Which team should handle this incident ticket?\nChoose one: application, infrastructure, database, backup.\nAnswer with the team name only.\n\nTicket: Node ran out of disk space and pods are being evicted'}]
[{'role': 'user', 'content': 'Which team should handle this incident ticket?\nChoose one: application, infrastructure, database, backup.\nAnswer with the team name only.\n\nTicket: Replication lag on the orders replica keeps growing'}]
[{'role': 'user', 'content': 'Which team should handle this incident ticket?\nChoose one: application, infrastructure, database, backup.\nAnswer with the team name only.\n\nTicket: Backup of the orders database failed: cannot connect to the database'}]


In [26]:
def ask(prompt, **settings):
    return local_client.chat.completions.create(model=LOCAL_MODEL, messages=prompt, **settings)

In [29]:
def answer(response):
    return response.choices[0].message.content

In [30]:
for ticket in tickets:
    messages = [{"role": "user", "content": template.format(ticket=ticket)}]
    llm_responses = ask(messages, temperature=0)
    final_answer = answer (llm_responses)
    print (final_answer)
    

Application
Infrastructure
Infrastructure
Infrastructure


In [32]:
tempalte = """Which team should handle this incident ticket?
Choose one: application, infrastructure, database, backup.
Answer with the team name only.

Ticket: Orders API throws NullPointerException on checkout
Team: application

Ticket: Kubernetes node NotReady in the prod cluster
Team: infrastructure

Ticket: Slow queries and lock waits on the customers database
Team: database

Ticket: Backup job for the crm database failed with a timeout
Team: backup

Ticket: {ticket}
Team:"""

for ticket in tickets:
    messages = [{"role": "user", "content": tempalte.format(ticket=ticket)}]
    print(ticket[:55].ljust(55), "->", answer(ask(messages, temperature=0)))

Checkout page returns HTTP 500 after the release        -> application
Node ran out of disk space and pods are being evicted   -> infrastructure
Replication lag on the orders replica keeps growing     -> infrastructure
Backup of the orders database failed: cannot connect to -> application


In [38]:
question = ("A cluster has 3 nodes. Each node has 16 GB of memory and the system uses 3 GB of it. "
            "Each pod needs 3 GB. How many pods can the cluster run in total?")

In [39]:
messages = [{"role": "user", "content": question + "\n\nAnswer with the number only."}]

In [40]:
messages

[{'role': 'user',
  'content': 'A cluster has 3 nodes. Each node has 16 GB of memory and the system uses 3 GB of it. Each pod needs 3 GB. How many pods can the cluster run in total?\n\nAnswer with the number only.'}]

In [41]:
print(answer(ask(messages, temperature=0)))

5


In [37]:
messages = [{"role": "user", "content": question + "\n\nThink step by step, then write the final answer on the last line."}]
print(answer(ask(messages, temperature=0)))

To find out how many pods can run in the cluster, we need to calculate the total amount of memory available and then subtract the memory used by the system.

1. Total memory available: 3 nodes * 16 GB/node = 48 GB
2. Memory used by the system: 3 GB
3. Total memory available after subtracting the system's memory: 48 GB - 3 GB = 45 GB
4. Memory required by each pod: 3 GB
5. Total number of pods that can run: 45 GB / 3 GB/pod = 15 pods

The cluster can run 15 pods in total.


In [42]:
def ask(prompt, **settings):
    return groq_client.chat.completions.create(model=GROQ_MODEL, messages=prompt, **settings)
    
def answer(response):
    return response.choices[0].message.content

In [43]:
question = ("A cluster has 3 nodes. Each node has 16 GB of memory and the system uses 3 GB of it. "
            "Each pod needs 3 GB. How many pods can the cluster run in total?")

In [44]:
messages = [{"role": "user", "content": question + "\n\nAnswer with the number only."}]

In [45]:
print(answer(ask(messages, temperature=0)))

12


In [49]:
messages = [{"role": "user", "content": question + "\n\nThink step by step, then write the final answer on the last line."}]
print(answer(ask(messages, temperature=0)))

**Step‑by‑step calculation**

1. **Memory per node:** 16 GB  
2. **Memory used by the system per node:** 3 GB  
3. **Memory available for pods per node:** 16 GB − 3 GB = **13 GB**  
4. **Memory required per pod:** 3 GB  
5. **Maximum pods per node:** ⌊13 GB ÷ 3 GB⌋ = **4 pods** (4 × 3 GB = 12 GB, 1 GB remains unused)  
6. **Number of nodes in the cluster:** 3  
7. **Total pods the cluster can run:** 4 pods/node × 3 nodes = **12 pods**

**Answer:**  
12 pods can be run in total.


In [56]:
messages = [{"role": "user", "content": "tell me a 2 line beautiful love story"}]
print(answer(ask(messages, temperature=1.2)))

In a rain‑kissed garden, she whispered a secret word that made his heart bloom like midnight roses; he answered with a single glance, and time folded into forever.  

Together they painted the horizon with laughter, each sunrise a promise that their love would outshine the stars.
